# Adversarial Structural Estimation on a Single Network

Linear-in-means MVP demo notebook following `docs/design_doc.md`.

## C1 - Imports, seeds, provenance capture

In [1]:
import csv
import hashlib
import json
import math
import os
import platform
import subprocess
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings(
    "ignore",
    message=r"`torch_geometric\.distributed` has been deprecated since 2\.7\.0.*",
    category=DeprecationWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r"`torch\.jit\.script` is deprecated\..*",
    category=DeprecationWarning,
)

import torch_geometric
from torch_geometric.utils import degree, from_networkx, k_hop_subgraph, to_undirected

print("torch:", torch.__version__)
print("torch_geometric:", torch_geometric.__version__)
print("networkx:", nx.__version__)
print("numpy:", np.__version__)

plt.style.use("seaborn-v0_8-whitegrid")

torch.manual_seed(42)
np.random.seed(42)
torch.set_default_dtype(torch.float32)

DEVICE = torch.device("cpu")
print("device:", DEVICE)

# torch.use_deterministic_algorithms(True)

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

# Import GAN components from dedicated modules
from generator import SCMGenerator
from discriminator import RootedMPNNDiscriminator

# Import utilities
from utils import (
    build_row_stochastic_W,
    compute_instance_noise_taus,
    extract_ego_batch,
)
from root_sampling import RootSampler, build_adjacency_from_edge_index, sample_roots_tensor

from visualization import plot_ground_truth_graph_outcomes


torch: 2.10.0+cpu
torch_geometric: 2.7.0
networkx: 3.6.1
numpy: 1.26.4
device: cpu


## C2 - Configuration (single source of truth)

In [2]:
from dataclasses import replace

from config import ExperimentConfig

cfg = ExperimentConfig.default()

# Baseline override (BA + uniform root sampling):
# cfg = replace(
#     cfg,
#     graph=replace(cfg.graph, graph_type="ba"),
#     training=replace(cfg.training, root_sampler_mode="uniform"),
# )

torch.manual_seed(cfg.graph.seed)
np.random.seed(cfg.graph.seed)

# True parameters
TRUE_BETA = cfg.true_params.beta
TRUE_GAMMA = cfg.true_params.gamma
TRUE_SIGMA_SQ = cfg.true_params.sigma_sq

# Graph
GRAPH_TYPE = cfg.graph.graph_type
N_NODES_TARGET = cfg.graph.n_nodes
BA_M = cfg.graph.ba_m
GRAPH_SEED = cfg.graph.seed

# Estimation
K = cfg.model.k
BATCH_SIZE = cfg.training.batch_size
N_DISC = cfg.training.n_disc
N_STEPS = cfg.training.n_steps
LR_D = cfg.training.lr_d
LR_G = cfg.training.lr_g
HIDDEN_DIM = cfg.model.hidden_dim
LOGIT_CLIP = cfg.model.logit_clip

# Picard iteration
BETA_CAP = cfg.model.beta_cap
PICARD_TOL = cfg.model.picard_tol
PICARD_MAX = cfg.model.picard_max

# Generator initialization
INIT_BETA = cfg.init_params.beta
INIT_GAMMA = cfg.init_params.gamma
INIT_LOG_SIGMA_SQ = cfg.init_params.log_sigma_sq

# Root sampling
ROOT_SAMPLER_MODE = cfg.training.resolved_root_sampler_mode()
ROOT_EXCLUSION_R = cfg.training.root_exclusion_r
DISJOINT_RESTARTS_K = cfg.training.resolved_disjoint_restarts_k()
DISJOINT_MIN_BATCH = cfg.training.resolved_disjoint_min_batch()
DISJOINT_RELAX_SEQUENCE = cfg.training.resolved_disjoint_relax_sequence()
DISJOINT_FALLBACK = cfg.training.disjoint_fallback
MIX_P_UNIFORM = cfg.training.mix_p_uniform
ROOT_SAMPLER_IS_DISJOINT = ROOT_SAMPLER_MODE != "uniform"

# Optional discriminator-input blur (instance noise)
INSTANCE_NOISE = cfg.instance_noise
if INSTANCE_NOISE.enabled and INSTANCE_NOISE.apply_to == "real_only":
    warnings.warn(
        "instance_noise.apply_to='real_only' is non-default and distorts only real discriminator targets.",
        RuntimeWarning,
    )

# Runtime diagnostics populated in later cells.
N_NODES = N_NODES_TARGET
graph_runtime_info = {}
sampling_calibration = {}
sampling_call_records = []
sampling_fallback_count = 0
sampling_call_counter = 0
last_sampling_info = None

RUN_ID = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
RUN_DIR = repo_root / "artifacts" / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

artifact_paths = []

print("run_id:", RUN_ID)
print("run_dir:", RUN_DIR)
print("graph_type:", GRAPH_TYPE)
print("root_sampler_mode:", ROOT_SAMPLER_MODE)
print("disjoint_restarts_k:", DISJOINT_RESTARTS_K)
print("disjoint_min_batch:", DISJOINT_MIN_BATCH)
print("disjoint_relax_sequence:", DISJOINT_RELAX_SEQUENCE)
print("instance_noise:", INSTANCE_NOISE)


run_id: 20260217_213448
run_dir: c:\Users\vitil\OneDrive\Desktop\adversarial_networks\artifacts\runs\20260217_213448
graph_type: lfr
root_sampler_mode: disjoint_best_of_k
disjoint_restarts_k: 50
disjoint_min_batch: 64
disjoint_relax_sequence: (3, 2)
instance_noise: InstanceNoiseConfig(enabled=True, tau_x0=1.0, tau_y0=1.2, schedule='linear', anneal_steps=2000, min_tau=0.0, apply_to='both')


## C3 - Graph generation, W matrix, and ego-cache

In [3]:
start_time = time.time()


def _build_lfr_graph_with_retries(graph_cfg, max_attempts: int = 3) -> tuple[nx.Graph, int]:
    last_error = None
    for attempt in range(max_attempts):
        attempt_seed = graph_cfg.seed + attempt
        lfr_kwargs = {
            "n": graph_cfg.n_nodes,
            "tau1": graph_cfg.lfr_tau1,
            "tau2": graph_cfg.lfr_tau2,
            "mu": graph_cfg.lfr_mu,
            "min_community": graph_cfg.lfr_min_community,
            "max_community": graph_cfg.lfr_max_community,
            "max_degree": graph_cfg.lfr_max_degree,
            "seed": attempt_seed,
            "max_iters": graph_cfg.lfr_max_iters,
        }
        if graph_cfg.lfr_average_degree is not None:
            lfr_kwargs["average_degree"] = graph_cfg.lfr_average_degree
        else:
            lfr_kwargs["min_degree"] = graph_cfg.lfr_min_degree

        try:
            graph_raw = nx.generators.community.LFR_benchmark_graph(**lfr_kwargs)
            return nx.Graph(graph_raw), attempt + 1
        except nx.ExceededMaxIterations as exc:
            last_error = exc
            warnings.warn(
                "LFR generation exceeded max_iters "
                f"(attempt {attempt + 1}/{max_attempts}, seed={attempt_seed}, "
                f"max_iters={graph_cfg.lfr_max_iters}). Retrying.",
                RuntimeWarning,
            )

    raise RuntimeError(
        f"LFR generation failed after {max_attempts} attempts "
        f"(max_iters={graph_cfg.lfr_max_iters})."
    ) from last_error


if GRAPH_TYPE == "ba":
    G_nx = nx.barabasi_albert_graph(n=N_NODES_TARGET, m=BA_M, seed=GRAPH_SEED)
    lfr_attempts = None
else:
    G_nx, lfr_attempts = _build_lfr_graph_with_retries(cfg.graph, max_attempts=3)

n_nodes_before_sanitize = G_nx.number_of_nodes()
n_selfloops_removed = nx.number_of_selfloops(G_nx)
if n_selfloops_removed > 0:
    G_nx.remove_edges_from(nx.selfloop_edges(G_nx))

if not nx.is_connected(G_nx):
    gcc_nodes = max(nx.connected_components(G_nx), key=len)
    gcc_fraction = len(gcc_nodes) / n_nodes_before_sanitize
    G_nx = G_nx.subgraph(gcc_nodes).copy()
else:
    gcc_fraction = 1.0

if set(G_nx.nodes()) != set(range(G_nx.number_of_nodes())):
    G_nx = nx.convert_node_labels_to_integers(G_nx, first_label=0, ordering="sorted")

isolates = [node for node, deg_val in G_nx.degree() if deg_val == 0]
if isolates:
    raise RuntimeError(
        f"Graph has {len(isolates)} isolates after sanitization; cannot build row-stochastic W."
    )

N_NODES = G_nx.number_of_nodes()
edge_pairs = np.asarray(list(G_nx.edges()), dtype=np.int64)
if edge_pairs.size == 0:
    raise RuntimeError("Graph has no edges after sanitization.")

edge_index = torch.as_tensor(edge_pairs.T, dtype=torch.long)
edge_index = to_undirected(edge_index, num_nodes=N_NODES).contiguous().to(DEVICE)
W = build_row_stochastic_W(edge_index=edge_index, num_nodes=N_NODES).to(DEVICE)

deg = degree(edge_index[0], num_nodes=N_NODES, dtype=torch.float32)

graph_runtime_info = {
    "graph_type": GRAPH_TYPE,
    "seed": int(GRAPH_SEED),
    "configured_n_nodes": int(N_NODES_TARGET),
    "n_nodes": int(N_NODES),
    "n_edges": int(G_nx.number_of_edges()),
    "degree_min": float(deg.min().item()),
    "degree_mean": float(deg.mean().item()),
    "degree_max": float(deg.max().item()),
    "n_selfloops_removed": int(n_selfloops_removed),
    "gcc_fraction": float(gcc_fraction),
}
if GRAPH_TYPE == "ba":
    graph_runtime_info["ba_m"] = int(BA_M)
else:
    graph_runtime_info["lfr_params"] = {
        "tau1": float(cfg.graph.lfr_tau1),
        "tau2": float(cfg.graph.lfr_tau2),
        "mu": float(cfg.graph.lfr_mu),
        "average_degree": (
            int(cfg.graph.lfr_average_degree)
            if cfg.graph.lfr_average_degree is not None
            else None
        ),
        "min_degree": (
            int(cfg.graph.lfr_min_degree)
            if cfg.graph.lfr_min_degree is not None
            else None
        ),
        "min_community": int(cfg.graph.lfr_min_community),
        "max_community": int(cfg.graph.lfr_max_community),
        "max_degree": (
            int(cfg.graph.lfr_max_degree) if cfg.graph.lfr_max_degree is not None else None
        ),
        "max_iters": int(cfg.graph.lfr_max_iters),
        "attempts": int(lfr_attempts),
    }

cache_start = time.time()
ego_cache = {}
for root in range(N_NODES):
    subset, sub_edge_index, mapping, _ = k_hop_subgraph(
        node_idx=root,
        num_hops=K,
        edge_index=edge_index,
        relabel_nodes=True,
        num_nodes=N_NODES,
    )
    ego_cache[root] = (
        subset.to(DEVICE),
        sub_edge_index.to(DEVICE),
        int(mapping.item()),
    )
cache_seconds = time.time() - cache_start

root_sampler_rng = np.random.default_rng(GRAPH_SEED + 1)
uniform_sampler = RootSampler(
    num_nodes=N_NODES,
    mode="uniform",
    rng=root_sampler_rng,
)

if ROOT_SAMPLER_IS_DISJOINT:
    adjacency = build_adjacency_from_edge_index(edge_index=edge_index.cpu(), num_nodes=N_NODES)
    root_sampler = RootSampler(
        num_nodes=N_NODES,
        mode=ROOT_SAMPLER_MODE,
        exclusion_r=ROOT_EXCLUSION_R,
        disjoint_restarts_k=DISJOINT_RESTARTS_K,
        disjoint_min_batch=DISJOINT_MIN_BATCH,
        disjoint_relax_sequence=DISJOINT_RELAX_SEQUENCE,
        disjoint_fallback=DISJOINT_FALLBACK,
        rng=root_sampler_rng,
        adjacency=adjacency,
    )

    calibration_rng = np.random.default_rng(GRAPH_SEED + 2)
    calibration_sampler = RootSampler(
        num_nodes=N_NODES,
        mode=ROOT_SAMPLER_MODE,
        exclusion_r=ROOT_EXCLUSION_R,
        disjoint_restarts_k=DISJOINT_RESTARTS_K,
        disjoint_min_batch=DISJOINT_MIN_BATCH,
        disjoint_relax_sequence=DISJOINT_RELAX_SEQUENCE,
        disjoint_fallback=DISJOINT_FALLBACK,
        rng=calibration_rng,
        adjacency=adjacency,
    )
    calibration_sizes = np.array(
        [
            calibration_sampler.sample(batch_size=N_NODES).achieved_size
            for _ in range(50)
        ],
        dtype=np.int32,
    )
    sampling_calibration = {
        "trials": int(calibration_sizes.size),
        "min": int(calibration_sizes.min()),
        "p10": float(np.quantile(calibration_sizes, 0.10)),
        "median": float(np.quantile(calibration_sizes, 0.50)),
        "p90": float(np.quantile(calibration_sizes, 0.90)),
        "max": int(calibration_sizes.max()),
    }

    print(
        f"{ROOT_SAMPLER_MODE} calibration: "
        f"min={sampling_calibration['min']} "
        f"p10={sampling_calibration['p10']:.1f} "
        f"median={sampling_calibration['median']:.1f} "
        f"p90={sampling_calibration['p90']:.1f} "
        f"max={sampling_calibration['max']}"
    )

    if sampling_calibration["p10"] < DISJOINT_MIN_BATCH:
        warnings.warn(
            "Disjoint calibration p10 is below disjoint_min_batch: "
            f"p10={sampling_calibration['p10']:.1f}, "
            f"disjoint_min_batch={DISJOINT_MIN_BATCH}.",
            RuntimeWarning,
        )
else:
    root_sampler = uniform_sampler

print(f"nodes: {graph_runtime_info['n_nodes']}")
print(f"edges (undirected): {graph_runtime_info['n_edges']}")
print(
    "degree mean/min/max: "
    f"{graph_runtime_info['degree_mean']:.3f} / "
    f"{graph_runtime_info['degree_min']:.0f} / "
    f"{graph_runtime_info['degree_max']:.0f}"
)
print(f"self-loops removed: {graph_runtime_info['n_selfloops_removed']}")
print(f"gcc_fraction: {graph_runtime_info['gcc_fraction']:.3f}")
print(f"ego-cache entries: {len(ego_cache)}")
print(f"ego-cache build time: {cache_seconds:.2f}s")
print(f"C3 total time: {time.time() - start_time:.2f}s")


disjoint_best_of_k calibration: min=3713 p10=3724.0 median=3741.5 p90=3759.3 max=3767
nodes: 249813
edges (undirected): 688630
degree mean/min/max: 5.513 / 1 / 102
self-loops removed: 9295
gcc_fraction: 0.999
ego-cache entries: 249813
ego-cache build time: 1476.64s
C3 total time: 2130.27s


## C4 - Simulate observed equilibrium

In [4]:
X = torch.randn(N_NODES, device=DEVICE)

true_generator = SCMGenerator(
    beta_cap=BETA_CAP,
    picard_tol=PICARD_TOL,
    picard_max=PICARD_MAX,
    init_beta=TRUE_BETA,
    init_gamma=TRUE_GAMMA,
    init_log_sigma_sq=math.log(TRUE_SIGMA_SQ),
).to(DEVICE)

with torch.no_grad():
    Y_obs = true_generator(W, X)
obs_picard_iters = true_generator.last_picard_iterations

norm_stats = {
    "mu_X": float(X.mean().item()),
    "sigma_X": float(X.std(unbiased=False).item()),
    "mu_Y": float(Y_obs.mean().item()),
    "sigma_Y": float(Y_obs.std(unbiased=False).item()),
}

print("observed Picard iterations:", obs_picard_iters)
print("normalization constants:", norm_stats)


observed Picard iterations: 15
normalization constants: {'mu_X': -0.00017465927521698177, 'sigma_X': 1.0027995109558105, 'mu_Y': -0.0077750785276293755, 'sigma_Y': 1.90252685546875}


## C5 - Exploratory data analysis (Figure 1, Table 1)

In [5]:
def write_csv_rows(path: Path, header: list[str], rows: list[list[object]]) -> None:
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(header)
        writer.writerows(rows)


deg_np = deg.cpu().numpy()
X_np = X.detach().cpu().numpy()
Y_obs_np = Y_obs.detach().cpu().numpy()
ols_slope, ols_intercept = np.polyfit(X_np, Y_obs_np, deg=1)

fig1, axes = plt.subplots(1, 3, figsize=(16, 4.8))

axes[0].hist(deg_np, bins=35, color="steelblue", alpha=0.85, edgecolor="black", linewidth=0.4)
axes[0].set_yscale("log")
axes[0].set_title("(a) Degree Distribution")
axes[0].set_xlabel("Degree")
axes[0].set_ylabel("Count (log scale)")

axes[1].hist(Y_obs_np, bins=40, color="steelblue", alpha=0.85, edgecolor="black", linewidth=0.4)
axes[1].set_title("(b) Observed Outcome Distribution")
axes[1].set_xlabel("Y_obs")
axes[1].set_ylabel("Count")

axes[2].scatter(X_np, Y_obs_np, s=10, alpha=0.25, color="steelblue", edgecolors="none")
x_line = np.linspace(X_np.min(), X_np.max(), 200)
axes[2].plot(x_line, ols_slope * x_line + ols_intercept, color="red", linestyle="--", linewidth=2, label=f"OLS slope={ols_slope:.3f}")
axes[2].set_title("(c) X vs Y_obs")
axes[2].set_xlabel("X")
axes[2].set_ylabel("Y_obs")
axes[2].legend(loc="best")

fig1.tight_layout()
fig1_path = RUN_DIR / "fig01_observed_data.png"
fig1.savefig(fig1_path, dpi=150, bbox_inches="tight")
plt.close(fig1)

rows_table1 = [
    ["Nodes", int(N_NODES)],
    ["Edges (undirected)", int(G_nx.number_of_edges())],
    ["Mean degree", float(deg.mean().item())],
    ["Max degree", float(deg.max().item())],
    ["Mean Y_obs", float(Y_obs.mean().item())],
    ["Std Y_obs", float(Y_obs.std(unbiased=False).item())],
    ["OLS slope (Y on X)", float(ols_slope)],
    ["Picard iterations", int(obs_picard_iters)],
]
tab1_path = RUN_DIR / "tab01_data_summary.csv"
write_csv_rows(tab1_path, ["Statistic", "Value"], rows_table1)

artifact_paths.extend([fig1_path, tab1_path])

plot_ground_truth_graph_outcomes(
    graph=G_nx,
    outcomes=Y_obs_np,
    save_path=RUN_DIR / "fig07_ground_truth_graph_outcomes.png",
    max_nodes=1200,
    layout_seed=GRAPH_SEED,
)
fig7_path = RUN_DIR / "fig07_ground_truth_graph_outcomes.png"
artifact_paths.append(fig7_path)
print("saved:", fig7_path.name)

print("Table 1")
for key, value in rows_table1:
    print(f"  {key}: {value}")


saved: fig07_ground_truth_graph_outcomes.png
Table 1
  Nodes: 249813
  Edges (undirected): 688630
  Mean degree: 5.513164043426514
  Max degree: 102.0
  Mean Y_obs: -0.0077750785276293755
  Std Y_obs: 1.90252685546875
  OLS slope (Y on X): 1.545248538466228
  Picard iterations: 15


## C6 - Instantiate models and optimizers

In [6]:
generator = SCMGenerator(
    beta_cap=BETA_CAP,
    picard_tol=PICARD_TOL,
    picard_max=PICARD_MAX,
    init_beta=INIT_BETA,
    init_gamma=INIT_GAMMA,
    init_log_sigma_sq=INIT_LOG_SIGMA_SQ,
).to(DEVICE)

discriminator = RootedMPNNDiscriminator(hidden_dim=HIDDEN_DIM, logit_clip=LOGIT_CLIP).to(DEVICE)

opt_d = torch.optim.Adam(discriminator.parameters(), lr=LR_D)
opt_g = torch.optim.Adam(generator.parameters(), lr=LR_G)

history = {
    "beta": [],
    "gamma": [],
    "sigma_sq": [],
    "loss_d": [],
    "loss_g": [],
    "tau_x": [],
    "tau_y": [],
}

num_gen_params = sum(param.numel() for param in generator.parameters())
num_disc_params = sum(param.numel() for param in discriminator.parameters())

print("generator parameter count:", num_gen_params)
print("discriminator parameter count:", num_disc_params)
print("initial theta:", generator.get_params())


generator parameter count: 3
discriminator parameter count: 17219
initial theta: {'beta': 0.0, 'gamma': 0.0, 'sigma_sq': 1.0}


## C7 - Training loop (detach discipline + logging)

In [7]:
def sample_roots(batch_size: int) -> torch.Tensor:
    global sampling_call_counter, sampling_fallback_count, last_sampling_info

    sampling_call_counter += 1

    if ROOT_SAMPLER_IS_DISJOINT and MIX_P_UNIFORM > 0.0 and root_sampler_rng.random() < MIX_P_UNIFORM:
        roots, result = sample_roots_tensor(
            sampler=uniform_sampler,
            batch_size=batch_size,
            device=DEVICE,
        )
        record = {
            "call_index": int(sampling_call_counter),
            "requested_size": int(batch_size),
            "achieved_size": int(batch_size),
            "mode": "uniform_mix",
            "r_used": 0,
            "attempts_used": int(result.attempts_used),
            "met_target": True,
            "fallback_reason": "mix_p_uniform",
        }
    else:
        roots, result = sample_roots_tensor(
            sampler=root_sampler,
            batch_size=batch_size,
            device=DEVICE,
        )
        record = {
            "call_index": int(sampling_call_counter),
            "requested_size": int(batch_size),
            "achieved_size": int(result.achieved_size),
            "mode": str(result.mode),
            "r_used": int(result.radius_used) if result.radius_used is not None else 0,
            "attempts_used": int(result.attempts_used),
            "met_target": bool(result.met_target),
            "fallback_reason": str(result.fallback_reason),
        }

    if ROOT_SAMPLER_IS_DISJOINT and record["fallback_reason"] not in {"", "mix_p_uniform"}:
        sampling_fallback_count += 1

    sampling_call_records.append(record)
    last_sampling_info = record
    return roots


train_start = time.time()
for step in range(1, N_STEPS + 1):
    # Mild generator LR decay to reduce late-stage drift while preserving TTUR baseline.
    if step in (1000, 1500):
        for group in opt_g.param_groups:
            group["lr"] *= 0.5

    discriminator.train()
    for param in discriminator.parameters():
        param.requires_grad_(True)

    # Reuse one detached equilibrium draw across D updates for lower Monte Carlo noise.
    with torch.no_grad():
        Y_sim_detached = generator(W, X)

    tau_x_step, tau_y_step = compute_instance_noise_taus(
        instance_noise=INSTANCE_NOISE,
        generator_step=step,
    )

    step_sampling_achieved = []
    step_sampling_attempts = []

    last_loss_d = float("nan")
    for _ in range(N_DISC):
        roots = sample_roots(BATCH_SIZE)
        step_sampling_achieved.append(int(last_sampling_info["achieved_size"]))
        step_sampling_attempts.append(int(last_sampling_info["attempts_used"]))

        batch_obs, root_idx_obs = extract_ego_batch(
            roots=roots,
            ego_cache=ego_cache,
            X=X,
            Y=Y_obs,
            norm_stats=norm_stats,
            instance_noise=INSTANCE_NOISE,
            generator_step=step,
            batch_role="real",
        )

        batch_sim, root_idx_sim = extract_ego_batch(
            roots=roots,
            ego_cache=ego_cache,
            X=X,
            Y=Y_sim_detached,
            norm_stats=norm_stats,
            instance_noise=INSTANCE_NOISE,
            generator_step=step,
            batch_role="fake",
        )

        opt_d.zero_grad(set_to_none=True)
        logits_real = discriminator(batch_obs.x, batch_obs.edge_index, root_idx_obs)
        logits_fake = discriminator(batch_sim.x, batch_sim.edge_index, root_idx_sim)
        loss_d = F.softplus(-logits_real).mean() + F.softplus(logits_fake).mean()
        loss_d.backward()
        opt_d.step()
        last_loss_d = float(loss_d.item())

    discriminator.train()
    for param in discriminator.parameters():
        param.requires_grad_(False)

    roots_g = sample_roots(BATCH_SIZE)
    step_sampling_achieved.append(int(last_sampling_info["achieved_size"]))
    step_sampling_attempts.append(int(last_sampling_info["attempts_used"]))

    Y_sim = generator(W, X)
    batch_sim_g, root_idx_sim_g = extract_ego_batch(
        roots=roots_g,
        ego_cache=ego_cache,
        X=X,
        Y=Y_sim,
        norm_stats=norm_stats,
        instance_noise=INSTANCE_NOISE,
        generator_step=step,
        batch_role="fake",
    )

    opt_g.zero_grad(set_to_none=True)
    logits_fake_g = discriminator(batch_sim_g.x, batch_sim_g.edge_index, root_idx_sim_g)
    loss_g = F.softplus(-logits_fake_g).mean()
    loss_g.backward()
    nn.utils.clip_grad_norm_(generator.parameters(), max_norm=5.0)
    opt_g.step()

    theta = generator.get_params()
    history["beta"].append(theta["beta"])
    history["gamma"].append(theta["gamma"])
    history["sigma_sq"].append(theta["sigma_sq"])
    history["loss_d"].append(last_loss_d)
    history["loss_g"].append(float(loss_g.item()))
    history["tau_x"].append(float(tau_x_step))
    history["tau_y"].append(float(tau_y_step))

    if step == 1 or step % 50 == 0:
        log_msg = (
            f"step {step:4d}/{N_STEPS} | "
            f"beta={theta['beta']:.4f} gamma={theta['gamma']:.4f} sigma_sq={theta['sigma_sq']:.4f} | "
            f"tau_x={tau_x_step:.4f} tau_y={tau_y_step:.4f} | "
            f"L_D={last_loss_d:.4f} L_G={loss_g.item():.4f}"
        )
        if ROOT_SAMPLER_IS_DISJOINT and step_sampling_achieved:
            achieved_min = int(np.min(step_sampling_achieved))
            achieved_mean = float(np.mean(step_sampling_achieved))
            tries_mean = float(np.mean(step_sampling_attempts))
            tries_max = int(np.max(step_sampling_attempts))
            log_msg += (
                f" | |S| min/mean={achieved_min}/{achieved_mean:.1f} of {BATCH_SIZE} "
                f"tries(mean/max)={tries_mean:.2f}/{tries_max}"
            )
        print(log_msg)

train_seconds = time.time() - train_start
print(f"training time: {train_seconds:.2f}s")


step    1/800 | beta=0.0040 gamma=0.0050 sigma_sq=1.0050 | tau_x=0.9995 tau_y=1.1994 | L_D=1.3677 L_G=0.6260 | |S| min/mean=64/64.0 of 64 tries(mean/max)=1.00/1
step   50/800 | beta=0.1280 gamma=0.2589 sigma_sq=1.3020 | tau_x=0.9750 tau_y=1.1700 | L_D=0.1417 L_G=4.6371 | |S| min/mean=64/64.0 of 64 tries(mean/max)=1.00/1
step  100/800 | beta=0.3676 gamma=0.5470 sigma_sq=1.7844 | tau_x=0.9500 tau_y=1.1400 | L_D=0.6233 L_G=4.8987 | |S| min/mean=64/64.0 of 64 tries(mean/max)=1.00/1
step  150/800 | beta=0.5171 gamma=0.8043 sigma_sq=2.2407 | tau_x=0.9250 tau_y=1.1100 | L_D=0.3504 L_G=3.0959 | |S| min/mean=64/64.0 of 64 tries(mean/max)=1.00/1
step  200/800 | beta=0.5672 gamma=1.0775 sigma_sq=2.0884 | tau_x=0.9000 tau_y=1.0800 | L_D=0.2807 L_G=2.4880 | |S| min/mean=64/64.0 of 64 tries(mean/max)=1.00/1
step  250/800 | beta=0.5273 gamma=1.2467 sigma_sq=1.5631 | tau_x=0.8750 tau_y=1.0500 | L_D=1.1424 L_G=0.8761 | |S| min/mean=64/64.0 of 64 tries(mean/max)=1.00/1
step  300/800 | beta=0.4582 gamma=

## C8 - Parameter convergence (Figure 2)

In [8]:
steps = np.arange(1, N_STEPS + 1)

fig2, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
series = [
    ("beta", TRUE_BETA, r"$\beta$"),
    ("gamma", TRUE_GAMMA, r"$\gamma$"),
    ("sigma_sq", TRUE_SIGMA_SQ, r"$\sigma^2$"),
]
for ax, (key, truth, label) in zip(axes, series):
    ax.plot(steps, history[key], color="steelblue", linewidth=1.5, label="Estimate")
    ax.axhline(truth, color="red", linestyle="--", linewidth=1.8, label="True")
    ax.set_ylabel(label)
    ax.legend(loc="best")
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Generator Step")
fig2.suptitle("Figure 2: Parameter Trajectories", y=1.01)
fig2.tight_layout()
fig2_path = RUN_DIR / "fig02_theta_convergence.png"
fig2.savefig(fig2_path, dpi=150, bbox_inches="tight")
plt.close(fig2)
artifact_paths.append(fig2_path)

print("saved:", fig2_path.name)


saved: fig02_theta_convergence.png


## C9 - Loss convergence (Figure 3)

In [9]:
def rolling_mean(values: list[float], window: int) -> tuple[np.ndarray, np.ndarray]:
    arr = np.asarray(values, dtype=np.float64)
    if arr.size < window:
        return np.arange(1, arr.size + 1), arr
    kernel = np.ones(window, dtype=np.float64) / float(window)
    smooth = np.convolve(arr, kernel, mode="valid")
    x_axis = np.arange(window, arr.size + 1)
    return x_axis, smooth


x_d, smooth_d = rolling_mean(history["loss_d"], window=50)
x_g, smooth_g = rolling_mean(history["loss_g"], window=50)

fig3, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(x_d, smooth_d, color="steelblue", linewidth=1.8)
axes[0].axhline(2.0 * math.log(2.0), color="red", linestyle="--", linewidth=1.5, label=r"$2\log 2$")
axes[0].set_title("(a) Discriminator Loss")
axes[0].set_xlabel("Generator Step")
axes[0].set_ylabel("Rolling Mean (window=50)")
axes[0].legend(loc="best")

axes[1].plot(x_g, smooth_g, color="steelblue", linewidth=1.8)
axes[1].axhline(math.log(2.0), color="red", linestyle="--", linewidth=1.5, label=r"$\log 2$")
axes[1].set_title("(b) Generator Loss")
axes[1].set_xlabel("Generator Step")
axes[1].set_ylabel("Rolling Mean (window=50)")
axes[1].legend(loc="best")

fig3.tight_layout()
fig3_path = RUN_DIR / "fig03_loss_convergence.png"
fig3.savefig(fig3_path, dpi=150, bbox_inches="tight")
plt.close(fig3)
artifact_paths.append(fig3_path)

print("saved:", fig3_path.name)


saved: fig03_loss_convergence.png


## C10 - Final estimation results (Table 2)

In [10]:
theta_hat = generator.get_params()

rows_table2 = [
    ["beta", TRUE_BETA, theta_hat["beta"], abs(theta_hat["beta"] - TRUE_BETA)],
    ["gamma", TRUE_GAMMA, theta_hat["gamma"], abs(theta_hat["gamma"] - TRUE_GAMMA)],
    ["sigma_sq", TRUE_SIGMA_SQ, theta_hat["sigma_sq"], abs(theta_hat["sigma_sq"] - TRUE_SIGMA_SQ)],
]

tab2_path = RUN_DIR / "tab02_estimation_results.csv"
write_csv_rows(
    tab2_path,
    ["Parameter", "True", "Estimated", "Absolute Error"],
    rows_table2,
)
artifact_paths.append(tab2_path)

print("Table 2")
for row in rows_table2:
    print(f"  {row[0]:>8s} | true={row[1]:.4f} est={row[2]:.4f} abs_err={row[3]:.4f}")


Table 2
      beta | true=0.4000 est=0.4096 abs_err=0.0096
     gamma | true=1.5000 est=1.5046 abs_err=0.0046
  sigma_sq | true=1.0000 est=1.0217 abs_err=0.0217


## C11 - Discriminator diagnostic: score distributions (Figure 4)

In [11]:
discriminator.eval()
with torch.no_grad():
    roots_diag = sample_roots(512)
    batch_real, root_real = extract_ego_batch(
        roots=roots_diag,
        ego_cache=ego_cache,
        X=X,
        Y=Y_obs,
        norm_stats=norm_stats,
        instance_noise=INSTANCE_NOISE,
        generator_step=N_STEPS,
        batch_role="real",
    )

    Y_sim_diag = generator(W, X)
    batch_fake, root_fake = extract_ego_batch(
        roots=roots_diag,
        ego_cache=ego_cache,
        X=X,
        Y=Y_sim_diag,
        norm_stats=norm_stats,
        instance_noise=INSTANCE_NOISE,
        generator_step=N_STEPS,
        batch_role="fake",
    )

    logits_real = discriminator(batch_real.x, batch_real.edge_index, root_real)
    logits_fake = discriminator(batch_fake.x, batch_fake.edge_index, root_fake)
    scores_real = torch.sigmoid(logits_real).cpu().numpy()
    scores_fake = torch.sigmoid(logits_fake).cpu().numpy()

fig4, ax = plt.subplots(figsize=(8.5, 4.5))
ax.hist(scores_real, bins=30, alpha=0.65, color="steelblue", label="Real", density=True)
ax.hist(scores_fake, bins=30, alpha=0.55, color="darkorange", label="Fake", density=True)
ax.axvline(0.5, color="red", linestyle="--", linewidth=1.4)
ax.set_title("Figure 4: Discriminator Scores")
ax.set_xlabel("D(score)")
ax.set_ylabel("Density")
ax.legend(loc="best")

fig4.tight_layout()
fig4_path = RUN_DIR / "fig04_discriminator_scores.png"
fig4.savefig(fig4_path, dpi=150, bbox_inches="tight")
plt.close(fig4)
artifact_paths.append(fig4_path)

print("saved:", fig4_path.name)


saved: fig04_discriminator_scores.png


## C12 - Simulated vs observed outcome distributions (Figure 5)

In [12]:
with torch.no_grad():
    Y_sim_final = generator(W, X)

Y_sim_np = Y_sim_final.detach().cpu().numpy()

fig5, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))

axes[0].hist(Y_obs_np, bins=40, alpha=0.6, color="steelblue", density=True, label="Y_obs")
axes[0].hist(Y_sim_np, bins=40, alpha=0.5, color="darkorange", density=True, label="Y_sim(theta_hat)")
axes[0].set_title("(a) Outcome Histograms")
axes[0].set_xlabel("Y")
axes[0].set_ylabel("Density")
axes[0].legend(loc="best")

q_grid = np.linspace(0.01, 0.99, 99)
q_obs = np.quantile(Y_obs_np, q_grid)
q_sim = np.quantile(Y_sim_np, q_grid)
min_q = float(min(q_obs.min(), q_sim.min()))
max_q = float(max(q_obs.max(), q_sim.max()))
axes[1].scatter(q_obs, q_sim, s=14, alpha=0.8, color="steelblue")
axes[1].plot([min_q, max_q], [min_q, max_q], color="red", linestyle="--", linewidth=1.5)
axes[1].set_title("(b) Q-Q Plot")
axes[1].set_xlabel("Observed Quantiles")
axes[1].set_ylabel("Simulated Quantiles")

fig5.tight_layout()
fig5_path = RUN_DIR / "fig05_Y_distributions.png"
fig5.savefig(fig5_path, dpi=150, bbox_inches="tight")
plt.close(fig5)
artifact_paths.append(fig5_path)

print("saved:", fig5_path.name)


saved: fig05_Y_distributions.png


## C13 - Convergence tail statistics (Table 3, Figure 6)

In [13]:
tail_window = 500

beta_tail = np.asarray(history["beta"][-tail_window:], dtype=np.float64)
gamma_tail = np.asarray(history["gamma"][-tail_window:], dtype=np.float64)
sigma_tail = np.asarray(history["sigma_sq"][-tail_window:], dtype=np.float64)

tail_rows = [
    ["beta", float(beta_tail.mean()), float(beta_tail.std(ddof=0)), TRUE_BETA],
    ["gamma", float(gamma_tail.mean()), float(gamma_tail.std(ddof=0)), TRUE_GAMMA],
    ["sigma_sq", float(sigma_tail.mean()), float(sigma_tail.std(ddof=0)), TRUE_SIGMA_SQ],
]

tail_std = {
    "beta": float(beta_tail.std(ddof=0)),
    "gamma": float(gamma_tail.std(ddof=0)),
    "sigma_sq": float(sigma_tail.std(ddof=0)),
}

tab3_path = RUN_DIR / "tab03_convergence_tail.csv"
write_csv_rows(
    tab3_path,
    ["Parameter", "Mean (last 500)", "Std (last 500)", "True"],
    tail_rows,
)
artifact_paths.append(tab3_path)

tail_steps = np.arange(N_STEPS - tail_window + 1, N_STEPS + 1)
fig6, axes = plt.subplots(3, 1, figsize=(10, 8.5), sharex=True)
for ax, key, truth, label in [
    (axes[0], "beta", TRUE_BETA, r"$\beta$"),
    (axes[1], "gamma", TRUE_GAMMA, r"$\gamma$"),
    (axes[2], "sigma_sq", TRUE_SIGMA_SQ, r"$\sigma^2$"),
]:
    ax.plot(tail_steps, history[key][-tail_window:], color="steelblue", linewidth=1.6)
    ax.axhline(truth, color="red", linestyle="--", linewidth=1.6)
    ax.set_ylabel(label)
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Generator Step")
fig6.suptitle("Figure 6: Tail Stability (last 500 steps)", y=1.01)
fig6.tight_layout()
fig6_path = RUN_DIR / "fig06_tail_stability.png"
fig6.savefig(fig6_path, dpi=150, bbox_inches="tight")
plt.close(fig6)
artifact_paths.append(fig6_path)

print("Table 3")
for row in tail_rows:
    print(f"  {row[0]:>8s} | mean={row[1]:.4f} std={row[2]:.4f} true={row[3]:.4f}")


Table 3
      beta | mean=0.3886 std=0.0277 true=0.4000
     gamma | mean=1.5048 std=0.0371 true=1.5000
  sigma_sq | mean=0.9998 std=0.0598 true=1.0000


## C14 - Save all artifacts and write run manifest

In [14]:
def sha256_file(path: Path) -> str:
    hasher = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


if ROOT_SAMPLER_IS_DISJOINT and sampling_call_records:
    sampling_stats_path = RUN_DIR / "sampling_roots_per_call.csv"
    with sampling_stats_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "call_index",
                "requested_size",
                "achieved_size",
                "mode",
                "r_used",
                "attempts_used",
                "met_target",
                "fallback_reason",
            ],
        )
        writer.writeheader()
        writer.writerows(sampling_call_records)
    artifact_paths.append(sampling_stats_path)

seen = set()
unique_artifacts = []
for path in artifact_paths:
    if path.name not in seen:
        unique_artifacts.append(path)
        seen.add(path.name)

try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=repo_root, text=True, stderr=subprocess.DEVNULL
    ).strip()
    git_status = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=repo_root, text=True, stderr=subprocess.DEVNULL
    )
    git_is_dirty = bool(git_status.strip())
except Exception:
    git_commit = None
    git_is_dirty = None

conda_lock_path = repo_root / "conda-lock.yml"
env_yml_path = repo_root / "environment.yml"

if ROOT_SAMPLER_IS_DISJOINT and sampling_call_records:
    achieved_arr = np.asarray([row["achieved_size"] for row in sampling_call_records], dtype=np.int32)
    attempts_arr = np.asarray([row["attempts_used"] for row in sampling_call_records], dtype=np.int32)
    sampling_runtime_stats = {
        "calls": int(achieved_arr.size),
        "min": int(achieved_arr.min()),
        "p10": float(np.quantile(achieved_arr, 0.10)),
        "median": float(np.quantile(achieved_arr, 0.50)),
        "p90": float(np.quantile(achieved_arr, 0.90)),
        "max": int(achieved_arr.max()),
        "attempts_mean": float(attempts_arr.mean()),
        "attempts_max": int(attempts_arr.max()),
        "fallback_count": int(sampling_fallback_count),
    }
else:
    sampling_runtime_stats = None

manifest = {
    "run_id": RUN_ID,
    "timestamp_utc": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
    "git": {"commit": git_commit, "is_dirty": git_is_dirty},
    "platform": {
        "os": platform.platform(),
        "python": platform.python_version(),
    },
    "versions": {
        "torch": torch.__version__,
        "torch_geometric": torch_geometric.__version__,
        "numpy": np.__version__,
        "networkx": nx.__version__,
    },
    "files": {
        "conda_lock_sha256": sha256_file(conda_lock_path) if conda_lock_path.exists() else None,
        "environment_yml_sha256": sha256_file(env_yml_path) if env_yml_path.exists() else None,
    },
    "inputs": {
        "graph": {
            "graph_type": cfg.graph.graph_type,
            "n_nodes": int(cfg.graph.n_nodes),
            "seed": int(cfg.graph.seed),
            "ba_m": int(cfg.graph.ba_m),
            "lfr_tau1": float(cfg.graph.lfr_tau1),
            "lfr_tau2": float(cfg.graph.lfr_tau2),
            "lfr_mu": float(cfg.graph.lfr_mu),
            "lfr_average_degree": (
                int(cfg.graph.lfr_average_degree)
                if cfg.graph.lfr_average_degree is not None
                else None
            ),
            "lfr_min_degree": (
                int(cfg.graph.lfr_min_degree) if cfg.graph.lfr_min_degree is not None else None
            ),
            "lfr_min_community": int(cfg.graph.lfr_min_community),
            "lfr_max_community": int(cfg.graph.lfr_max_community),
            "lfr_max_degree": (
                int(cfg.graph.lfr_max_degree) if cfg.graph.lfr_max_degree is not None else None
            ),
            "lfr_max_iters": int(cfg.graph.lfr_max_iters),
            "realized": graph_runtime_info,
        },
        "sampling": {
            "root_sampler_mode": ROOT_SAMPLER_MODE,
            "root_exclusion_r": int(ROOT_EXCLUSION_R),
            "disjoint_restarts_k": int(DISJOINT_RESTARTS_K),
            "disjoint_min_batch": int(DISJOINT_MIN_BATCH),
            "disjoint_relax_sequence": [int(v) for v in DISJOINT_RELAX_SEQUENCE],
            "disjoint_fallback": str(DISJOINT_FALLBACK),
            "mix_p_uniform": float(MIX_P_UNIFORM),
            "calibration": sampling_calibration if ROOT_SAMPLER_IS_DISJOINT else None,
            "runtime": sampling_runtime_stats,
        },
        "instance_noise": {
            "enabled": bool(INSTANCE_NOISE.enabled),
            "tau_x0": float(INSTANCE_NOISE.tau_x0),
            "tau_y0": float(INSTANCE_NOISE.tau_y0),
            "schedule": str(INSTANCE_NOISE.schedule),
            "anneal_steps": int(INSTANCE_NOISE.anneal_steps),
            "min_tau": float(INSTANCE_NOISE.min_tau),
            "apply_to": str(INSTANCE_NOISE.apply_to),
            "final_tau_x": float(history["tau_x"][-1]) if history["tau_x"] else 0.0,
            "final_tau_y": float(history["tau_y"][-1]) if history["tau_y"] else 0.0,
        },
        "model": {
            "k": K,
            "beta_cap": BETA_CAP,
            "picard_tol": PICARD_TOL,
            "picard_max": PICARD_MAX,
            "hidden_dim": HIDDEN_DIM,
            "logit_clip": LOGIT_CLIP,
        },
        "training": {
            "steps": N_STEPS,
            "n_disc": N_DISC,
            "batch_size": BATCH_SIZE,
            "lr_d": LR_D,
            "lr_g": LR_G,
        },
    },
    "truth": {
        "beta0": TRUE_BETA,
        "gamma0": TRUE_GAMMA,
        "sigma_sq0": TRUE_SIGMA_SQ,
    },
    "estimate": {
        "beta_hat": float(theta_hat["beta"]),
        "gamma_hat": float(theta_hat["gamma"]),
        "sigma_sq_hat": float(theta_hat["sigma_sq"]),
    },
    "diagnostics": {"tail_std": tail_std},
    "artifacts": [
        {"path": path.name, "sha256": sha256_file(path)}
        for path in unique_artifacts
    ],
}

manifest_path = RUN_DIR / "run_manifest.json"
with manifest_path.open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

print("Artifacts written:")
for path in sorted(RUN_DIR.glob("*")):
    print(f"  {path.name}: {path.stat().st_size} bytes")


Artifacts written:
  fig01_observed_data.png: 223893 bytes
  fig02_theta_convergence.png: 121429 bytes
  fig03_loss_convergence.png: 89271 bytes
  fig04_discriminator_scores.png: 40440 bytes
  fig05_Y_distributions.png: 90901 bytes
  fig06_tail_stability.png: 109721 bytes
  fig07_ground_truth_graph_outcomes.png: 185028 bytes
  run_manifest.json: 4934 bytes
  sampling_roots_per_call.csv: 195830 bytes
  tab01_data_summary.csv: 230 bytes
  tab02_estimation_results.csv: 194 bytes
  tab03_convergence_tail.csv: 202 bytes
